# Persistent car parts: reproduce the real-video prototype

Tag a car's front-door panel and front window on frame 0, propagate those IDs, then recolor either saved predicted part without rerunning the model. This notebook runs **SAM 2.1 Hiera Tiny**, per-part Lucas–Kanade affine flow, and a shared-plane homography/SIFT baseline on the two real DAVIS clips.

Choose **Runtime → Change runtime type → T4 GPU**, then run the cells in order. The notebook downloads official footage and model files. Its source and checkpoint are pinned; it never trains a model. Keep Colab's matching PyTorch and torchvision installation. The default runs both clips; set `SEQUENCES` to one name for a shorter run.

**Interpretation:** frame-zero masks and later scoring polygons are approximate assistant-authored annotations, not official DAVIS part labels or human-verified ground truth. Later polygons reach only the scorer. These clips may overlap SAM 2 training data and do not test full target occlusion. Recoloring is deterministic compositing, not generative video editing or face-identity preservation.

This repository copy has **empty outputs**. It is a reproduction recipe, not evidence that this fresh notebook has been GPU-executed. Recorded results are separate files under `run_2026-10-09/`.


In [ ]:
from pathlib import Path
import json, sys, subprocess, hashlib, platform, shutil
from datetime import datetime, timezone

# This single revision pins all experiment code, annotations and source manifests.
REPO_REV = "f9e2947b94f50bfd610e3854f12132491d2940a8"
REPO_URL = "https://github.com/skbdps/vjepa-poc.git"
REPO = Path("/content/vjepa-poc-real-parts")
RUN_ROOT = Path("/content/real_part_consistency_run")
SEQUENCES = ("car-roundabout", "car-shadow")
FPS = 12

assert SEQUENCES and set(SEQUENCES) <= {"car-roundabout", "car-shadow"}
assert len(SEQUENCES) == len(set(SEQUENCES))
if not (REPO / ".git").exists():
    subprocess.run(["git", "clone", "--filter=blob:none", "--no-checkout", REPO_URL, str(REPO)], check=True)
subprocess.run(["git", "-C", str(REPO), "fetch", "origin", REPO_REV], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", REPO_REV], check=True)
actual_rev = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
assert actual_rev == REPO_REV
EXP = REPO / "experiments/day4"
sys.path.insert(0, str(EXP))
RUN_ROOT.mkdir(parents=True, exist_ok=True)
print("Experiment revision:", actual_rev)


## Install the pinned specialist

The helper checks out the official SAM 2 revision below, disables its optional CUDA extension, and downloads the official Tiny checkpoint. The optional postprocessing extension is not used by this experiment. Checkpoint SHA-256 is verified against the recorded run. Stop if the hash differs; do not substitute a different model silently.


In [ ]:
if shutil.which("ffmpeg") is None:
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", "ffmpeg"], check=True)
# Reuse available Colab packages to avoid replacing already imported modules.
from importlib.util import find_spec
packages = {"numpy": "numpy>=1.24", "cv2": "opencv-python-headless==4.11.0.86", "PIL": "pillow"}
missing = [package for module, package in packages.items() if find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
import torch
assert torch.cuda.is_available(), "Select a Colab GPU runtime before proceeding."
import numpy as np
import cv2
import real_video

SAM2_REV = "2b90b9f5ceec907a1c18123530e92e794ad901a4"
EXPECTED_CHECKPOINT_SHA256 = "7402e0d864fa82708a20fbd15bc84245c2f26dff0eb43a4b5b93452deb34be69"
assert real_video.SAM2_REVISION == SAM2_REV
checkpoint = real_video.prepare_sam2(root="/content/sam2", checkpoint_dir="/content/checkpoints")
assert real_video.sha256_file(checkpoint) == EXPECTED_CHECKPOINT_SHA256
assert subprocess.check_output(["git", "-C", "/content/sam2", "rev-parse", "HEAD"], text=True).strip() == SAM2_REV
from sam2.build_sam import build_sam2_video_predictor
predictor = build_sam2_video_predictor(real_video.SAM2_CONFIG, checkpoint,
                                      device="cuda", apply_postprocessing=False)
print("GPU:", torch.cuda.get_device_name())
print("SAM 2 revision:", SAM2_REV)
print("Checkpoint SHA-256:", EXPECTED_CHECKPOINT_SHA256)
print("Inference: float32; frame-zero prompts; no later corrections")


## Download exactly the recorded source frames

Every image URL and expected hash comes from the checked-out source manifest. Existing files are reused only after hash and image-size checks. `car-roundabout` has 64 selected frames; `car-shadow` has 40. The manifest's historical local path is ignored. Footage attribution and license notes are copied into the exported run.


In [ ]:
from concurrent.futures import ThreadPoolExecutor
from urllib.request import urlopen
from PIL import Image

def download_from_manifest(sequence):
    manifest_path = EXP / f"{sequence.replace('-', '_')}_source_manifest.json"
    manifest = json.loads(manifest_path.read_text())
    rows = manifest["frames"]
    assert manifest["sequence"] == sequence
    assert len(rows) == manifest["frame_count"]
    assert [r["frame_index"] for r in rows] == list(range(len(rows)))
    folder = RUN_ROOT / "inputs" / sequence
    frames_dir = folder / "frames"
    frames_dir.mkdir(parents=True, exist_ok=True)
    expected_names = {f"{r['frame_index']:05d}.jpg" for r in rows}
    assert not ({p.name for p in frames_dir.glob("*.jpg")} - expected_names), "Unexpected old frames"
    def fetch(row):
        path = frames_dir / f"{row['frame_index']:05d}.jpg"
        if not path.exists() or real_video.sha256_file(path) != row["sha256"]:
            for attempt in range(3):
                try:
                    with urlopen(row["url"], timeout=90) as response:
                        data = response.read()
                    assert hashlib.sha256(data).hexdigest() == row["sha256"], f"Source changed: {row['url']}"
                    temporary = path.with_suffix(".part")
                    temporary.write_bytes(data)
                    temporary.replace(path)
                    break
                except Exception:
                    if attempt == 2:
                        raise
        assert real_video.sha256_file(path) == row["sha256"]
        with Image.open(path) as image:
            assert list(image.size) == row["size_wh"]
            image.verify()
        return path
    with ThreadPoolExecutor(max_workers=4) as pool:
        downloaded = list(pool.map(fetch, rows))
    manifest["frames_dir"] = str(frames_dir.relative_to(RUN_ROOT))
    real_video.write_json(folder / "source_manifest.json", manifest)
    shutil.copy2(EXP / f"{sequence.replace('-', '_')}_annotations.json", folder / "annotations.json")
    print(sequence, len(downloaded), "frames verified")
    return frames_dir

frames_by_sequence = {sequence: download_from_manifest(sequence) for sequence in SEQUENCES}
shutil.copy2(EXP / "ATTRIBUTION.md", RUN_ROOT / "ATTRIBUTION.md")


## Run all three trackers and score once

`real_video.run()` supplies **only first-frame masks** to SAM 2, per-part affine flow and shared homography. It saves predicted masks, the persistent part registry, sparse scores, the selected-door edit and a comparison video. The later annotation frames are used solely to score predictions. Scores are diagnostic mean intersection-over-union (IoU), not a general identity benchmark.


In [ ]:
import contextlib
from part_editor import self_test

print("Compositing checks:", self_test())
run_manifest = {
    "started_utc": datetime.now(timezone.utc).isoformat(),
    "repo_url": REPO_URL, "repo_revision": REPO_REV,
    "sam2_revision": SAM2_REV, "checkpoint_url": real_video.SAM2_CHECKPOINT,
    "checkpoint_sha256": EXPECTED_CHECKPOINT_SHA256,
    "gpu": torch.cuda.get_device_name(), "python": platform.python_version(),
    "torch": torch.__version__, "cuda": torch.version.cuda,
    "numpy": np.__version__, "opencv": cv2.__version__,
    "precision": "float32", "postprocessing": False,
    "sequences": list(SEQUENCES), "preview_fps": FPS,
    "prompt_frames": [0], "tracker_correction_frames": [],
    "annotation_scope": "approximate assistant-authored diagnostic polygons; not human ground truth",
}
real_video.write_json(RUN_ROOT / "run_manifest.json", run_manifest)
reports = {}
for sequence in SEQUENCES:
    torch.manual_seed(0)
    np.random.seed(0)
    cv2.setRNGSeed(0)
    destination = RUN_ROOT / "results" / sequence
    destination.mkdir(parents=True, exist_ok=True)
    annotation_path = RUN_ROOT / "inputs" / sequence / "annotations.json"
    print("Running", sequence, flush=True)
    with (destination / "run.log").open("w") as log, contextlib.redirect_stdout(log):
        reports[sequence], timing = real_video.run(
            frames_by_sequence[sequence], annotation_path, destination,
            checkpoint=checkpoint, predictor=predictor)
    print(sequence, "SAM 2 propagation seconds:", round(timing["seconds"], 2))
    for method, result in reports[sequence].items():
        if "mean_iou" in result:
            print(f"  {method}: IoU={result['mean_iou']:.4f}, scored part-frames={result['scored_part_frames']}")
run_manifest["completed_utc"] = datetime.now(timezone.utc).isoformat()
real_video.write_json(RUN_ROOT / "run_manifest.json", run_manifest)


## Reuse the masks: change parts and colors

The same registry ID selects the same propagated part on every frame. Recoloring subtracts **all unselected predicted masks**, protecting their overlap pixels as well. A wrong predicted boundary can still recolor wrong real-world pixels. The exact preservation guarantee applies to RGB arrays before lossy MP4 encoding.

The editor embeds a compressed native-size MP4 and lossless mask RLE. It can be downloaded and opened by double-clicking the HTML, with no server or internet connection. The browser can switch parts, color, strength, time, outlines and labels. PNG export saves the current edited frame without diagnostic overlays.


In [ ]:
from part_editor import replay
from build_demo import build_demo
from IPython.display import display, Markdown, Video, HTML

for sequence in SEQUENCES:
    destination = RUN_ROOT / "results" / sequence
    registry = json.loads((destination / "part_registry.json").read_text())
    window_id = next(int(p["id"]) for p in registry["parts"] if p["name"] == "front_window")
    replay(frames_by_sequence[sequence], destination / "sam2_masks.npz",
           destination / "window_amber.mp4", target_id=window_id,
           color_rgb=(240, 180, 35), strength=.75, fps=FPS)
    built = build_demo(frames_by_sequence[sequence], destination / "sam2_masks.npz",
                      destination / "part_registry.json", destination / "interactive_part_editor.html",
                      fps=FPS, results_file=destination / "results.json")
    real_video.write_json(destination / "editor_build_manifest.json", built)
    display(Markdown(f"### {sequence}: selected door recolor"))
    display(Video(str(destination / "door_recolor.mp4"), embed=True, width=640))
    display(Markdown(f"### {sequence}: interactive part editor"))
    display(HTML(Path(built["output"]).read_text()))
    print("Offline editor:", built["output"], "bytes:", built["bytes"])


## Download the complete run

The archive contains original verified input frames, sparse annotations, source manifests, all three predicted-mask files, part registries, JSON metrics, logs, MP4s and offline HTML editors. It excludes model weights. Keep the ZIP or copy it to your own storage before Colab's temporary runtime is removed. Repository notebooks should be committed without outputs; preserve measured results separately.

Footage and modified previews follow the conservative **CC BY-NC 4.0** interpretation documented in `ATTRIBUTION.md`. SAM 2 code and checkpoints use Apache 2.0. Neither source endorses this prototype.


In [ ]:
from google.colab import files

inventory = []
for path in sorted(RUN_ROOT.rglob("*")):
    if path.is_file() and path.name != "artifact_inventory.json":
        inventory.append({"path": str(path.relative_to(RUN_ROOT)),
                          "bytes": path.stat().st_size,
                          "sha256": real_video.sha256_file(path)})
real_video.write_json(RUN_ROOT / "artifact_inventory.json", inventory)
archive = shutil.make_archive("/content/Real_Part_Consistency_Run", "zip", root_dir=RUN_ROOT)
print("Download:", archive, f"({Path(archive).stat().st_size / 2**20:.1f} MiB)")
files.download(archive)
